# Retail Sales Trend Analysis

## Project Overview
This notebook analyzes retail sales performance over time using the Sample Superstore dataset already available in this workspace. The focus is practical: identify trend direction, seasonality, strong and weak periods, leading categories, and operational signals that a retail team could actually use.

## Learning Objectives
- Build a time-based retail sales analysis workflow from raw transactional data.
- Separate overall revenue growth from seasonality and short-term noise.
- Compare category and sub-category performance using revenue, profit, and quantity together.
- Inspect discount and shipping-related patterns as potential revenue and profitability drivers.
- Translate descriptive analysis into cautious, actionable business recommendations.

## Business / Research Problem
Retail leaders need to understand when sales are strongest, when performance softens, and which product lines drive growth versus margin risk. Without that baseline, inventory, promotion, and merchandising decisions stay reactive.

## Why This Analysis Matters
Trend and seasonality analysis helps teams plan staffing, campaign timing, product emphasis, and discount strategy. Category analysis helps distinguish between revenue generators and margin contributors, which is essential for better pricing and assortment decisions.

## Dataset Overview
The notebook uses the repo-local `Sample - Superstore.xls` workbook. The verified schema includes retail order fields such as `Order Date`, `Ship Date`, `Category`, `Sub-Category`, `Sales`, `Quantity`, `Discount`, and `Profit`, which makes it a strong fit for time-based and category-based retail analysis.

## Dataset Source and License Notes
This workbook is a widely circulated Sample Superstore training dataset and is already present in this repository. Because this dataset appears in many mirrors, confirm the license and redistribution terms of the specific source copy you use outside this workspace. This notebook only analyzes the local workbook already available here.

## Environment Setup
The next cell installs only the packages required to run this notebook from a clean Python environment. The workbook is in `.xls` format, so `xlrd` is required for reading it with pandas.

In [1]:
import importlib
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "xlrd": "xlrd",
}

def ensure_package(package_name, import_name=None):
    module_name = import_name or package_name
    try:
        importlib.import_module(module_name)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

for package_name, import_name in REQUIRED_PACKAGES.items():
    ensure_package(package_name, import_name)

print("Environment setup complete.")

Environment setup complete.


## Imports
These imports cover path handling, data validation, aggregation, and plotting. The notebook keeps dependencies minimal so the analysis stays easy to rerun.

In [2]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
np.random.seed(42)

## Configuration / Constants
This block locates the workspace root, points to the workbook path, and defines the schema the notebook expects. Keeping these assumptions in one cell makes the analysis easier to audit later.

In [3]:
def locate_workspace_root(start_path):
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "README.md").exists():
            return candidate
    return start_path

WORKSPACE_ROOT = locate_workspace_root(Path.cwd())
DATASET_PATH = WORKSPACE_ROOT / "Time Series Analysis" / "Time Series Forecasting" / "Sample - Superstore.xls"
EXPECTED_COLUMNS = [
    "Row ID", "Order ID", "Order Date", "Ship Date", "Ship Mode", "Customer ID",
    "Customer Name", "Segment", "Country", "City", "State", "Postal Code",
    "Region", "Product ID", "Category", "Sub-Category", "Product Name",
    "Sales", "Quantity", "Discount", "Profit"
]

config_preview = {
    "workspace_root": str(WORKSPACE_ROOT),
    "dataset_path": str(DATASET_PATH),
    "expected_column_count": len(EXPECTED_COLUMNS),
}
print(json.dumps(config_preview, indent=2))

{
  "workspace_root": "E:\\Github\\Machine-Learning-Projects",
  "dataset_path": "E:\\Github\\Machine-Learning-Projects\\Time Series Analysis\\Time Series Forecasting\\Sample - Superstore.xls",
  "expected_column_count": 21
}


## Dataset Loading
The dataset is loaded from a repo-local workbook. If the file is missing, the notebook raises a clear error with the expected path instead of failing later in less obvious ways.

In [4]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Expected workbook not found at: {DATASET_PATH}")

raw_df = pd.read_excel(DATASET_PATH)
print(f"Raw shape: {raw_df.shape}")
display(raw_df.head())

Raw shape: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## Data Validation Checks
Before interpreting trends, this block verifies the schema, checks date parsing, reviews nulls, and looks for duplicate rows. This is the lowest-cost way to avoid drawing conclusions from broken inputs.

In [5]:
missing_columns = sorted(set(EXPECTED_COLUMNS) - set(raw_df.columns))
extra_columns = sorted(set(raw_df.columns) - set(EXPECTED_COLUMNS))

validation_df = raw_df.copy()
validation_df["Order Date"] = pd.to_datetime(validation_df["Order Date"], errors="coerce")
validation_df["Ship Date"] = pd.to_datetime(validation_df["Ship Date"], errors="coerce")

validation_report = pd.Series({
    "row_count": int(len(raw_df)),
    "missing_columns": missing_columns,
    "extra_columns": extra_columns,
    "invalid_order_dates": int(validation_df["Order Date"].isna().sum()),
    "invalid_ship_dates": int(validation_df["Ship Date"].isna().sum()),
    "duplicate_full_rows": int(raw_df.duplicated().sum()),
    "duplicate_order_product_rows": int(raw_df.duplicated(subset=["Order ID", "Product ID"]).sum()),
    "unique_orders": int(raw_df["Order ID"].nunique()),
    "unique_products": int(raw_df["Product ID"].nunique()),
    "unique_categories": int(raw_df["Category"].nunique()),
    "unique_subcategories": int(raw_df["Sub-Category"].nunique()),
}).to_frame(name="value")

missing_summary = raw_df.isna().sum().sort_values(ascending=False).to_frame(name="missing_count")

display(validation_report)
display(missing_summary)

,value
row_count,9994
missing_columns,[]
extra_columns,[]
invalid_order_dates,0
invalid_ship_dates,0
duplicate_full_rows,0
duplicate_order_product_rows,8
unique_orders,5009
unique_products,1862
unique_categories,3


,missing_count
Row ID,0
Order ID,0
Order Date,0
Ship Date,0
Ship Mode,0
Customer ID,0
Customer Name,0
Segment,0
Country,0
City,0


## Data Cleaning And Feature Preparation
This cleaning step keeps the analysis conservative. It removes duplicate rows, parses dates explicitly, standardizes key string columns, and adds time-based fields needed for trend, seasonality, and moving-average analysis.

In [6]:
df = raw_df.copy()
df = df.drop_duplicates().copy()
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], errors="coerce")
df = df.dropna(subset=["Order Date", "Category", "Sub-Category", "Sales", "Profit", "Quantity"]).copy()

string_columns = ["Category", "Sub-Category", "Segment", "Region", "Ship Mode", "State", "City", "Product Name"]
for column in string_columns:
    df[column] = df[column].astype(str).str.strip()

df["Sales"] = pd.to_numeric(df["Sales"], errors="coerce")
df["Profit"] = pd.to_numeric(df["Profit"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Discount"] = pd.to_numeric(df["Discount"], errors="coerce")
df = df.dropna(subset=["Sales", "Profit", "Quantity", "Discount"]).copy()

df["Order Year"] = df["Order Date"].dt.year
df["Order Month"] = df["Order Date"].dt.month
df["Order Month Name"] = df["Order Date"].dt.month_name()
df["Order Quarter"] = df["Order Date"].dt.to_period("Q").astype(str)
df["Order Weekday"] = df["Order Date"].dt.day_name()
df["YearMonth"] = df["Order Date"].dt.to_period("M").astype(str)
df["Ship Delay Days"] = (df["Ship Date"] - df["Order Date"]).dt.days
df["Profit Margin Pct"] = np.where(df["Sales"] != 0, (df["Profit"] / df["Sales"]) * 100, np.nan)
df["Discount Bucket"] = pd.cut(df["Discount"], bins=[-0.001, 0, 0.1, 0.2, 0.4, 1.0], labels=["0%", "0-10%", "10-20%", "20-40%", "40%+"])

cleaning_report = pd.Series({
    "clean_rows": int(len(df)),
    "date_start": str(df["Order Date"].min().date()),
    "date_end": str(df["Order Date"].max().date()),
    "total_sales": float(df["Sales"].sum()),
    "total_profit": float(df["Profit"].sum()),
    "average_order_line_sales": float(df["Sales"].mean()),
}).to_frame(name="value")

display(cleaning_report)
display(df.head())

,value
clean_rows,9994
date_start,2014-01-03
date_end,2017-12-30
total_sales,2297200.8603
total_profit,286397.0217
average_order_line_sales,229.858001


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Order Year,Order Month,Order Month Name,Order Quarter,Order Weekday,YearMonth,Ship Delay Days,Profit Margin Pct,Discount Bucket
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,2016,11,November,2016Q4,Tuesday,2016-11,3,16.00,0%
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,2016,11,November,2016Q4,Tuesday,2016-11,3,30.00,0%
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,2016,6,June,2016Q2,Sunday,2016-06,4,47.00,0%
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,2015,10,October,2015Q4,Sunday,2015-10,7,-40.00,40%+
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,2015,10,October,2015Q4,Sunday,2015-10,7,11.25,10-20%


## Exploratory Data AnalysisThis section aggregates sales and profit over time so we can see long-run direction without getting lost in row-level transaction noise. It also helps separate overall growth from temporary spikes and dips.

In [7]:
daily_trend = df.groupby("Order Date", as_index=False).agg(
    daily_sales=("Sales", "sum"),
    daily_profit=("Profit", "sum"),
    order_lines=("Order ID", "size"),
)
daily_trend["sales_ma_30"] = daily_trend["daily_sales"].rolling(30, min_periods=7).mean()
daily_trend["sales_ma_90"] = daily_trend["daily_sales"].rolling(90, min_periods=30).mean()

monthly_trend = df.groupby("YearMonth", as_index=False).agg(
    monthly_sales=("Sales", "sum"),
    monthly_profit=("Profit", "sum"),
    monthly_quantity=("Quantity", "sum"),
    monthly_orders=("Order ID", "nunique"),
)
monthly_trend["YearMonthDate"] = pd.to_datetime(monthly_trend["YearMonth"] + "-01")

fig, axes = plt.subplots(3, 1, figsize=(16, 14), sharex=False)
axes[0].plot(daily_trend["Order Date"], daily_trend["daily_sales"], color="lightsteelblue", label="Daily sales")
axes[0].plot(daily_trend["Order Date"], daily_trend["sales_ma_30"], color="navy", linewidth=2, label="30-day moving average")
axes[0].plot(daily_trend["Order Date"], daily_trend["sales_ma_90"], color="darkorange", linewidth=2, label="90-day moving average")
axes[0].set_title("Daily Sales With Moving Averages")
axes[0].legend()

axes[1].plot(monthly_trend["YearMonthDate"], monthly_trend["monthly_sales"], marker="o", color="teal")
axes[1].set_title("Monthly Sales Trend")
axes[1].set_ylabel("Sales")

axes[2].plot(monthly_trend["YearMonthDate"], monthly_trend["monthly_profit"], marker="o", color="firebrick")
axes[2].set_title("Monthly Profit Trend")
axes[2].set_ylabel("Profit")

plt.tight_layout()
plt.show()

display(monthly_trend.head())

C:\Users\ahmad\AppData\Local\Temp\ipykernel_45160\567291180.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,YearMonth,monthly_sales,monthly_profit,monthly_quantity,monthly_orders,YearMonthDate
0,2014-01,14236.895,2450.1907,284,32,2014-01-01
1,2014-02,4519.892,862.3084,159,28,2014-02-01
2,2014-03,55691.009,498.7299,585,71,2014-03-01
3,2014-04,28295.345,3488.8352,536,66,2014-04-01
4,2014-05,23648.287,2738.7096,466,69,2014-05-01


## Univariate AnalysisTrend alone is not enough. This block checks whether certain months or weekdays consistently outperform others and identifies weak periods by both sales and profit so revenue strength is not confused with healthy margin behavior.

In [8]:
month_order = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

monthly_seasonality = df.groupby("Order Month Name", as_index=False).agg(
    average_sales=("Sales", "mean"),
    total_sales=("Sales", "sum"),
    total_profit=("Profit", "sum"),
)
monthly_seasonality["Order Month Name"] = pd.Categorical(monthly_seasonality["Order Month Name"], categories=month_order, ordered=True)
monthly_seasonality = monthly_seasonality.sort_values("Order Month Name")

weekday_seasonality = df.groupby("Order Weekday", as_index=False).agg(
    average_sales=("Sales", "mean"),
    total_sales=("Sales", "sum"),
    total_profit=("Profit", "sum"),
)
weekday_seasonality["Order Weekday"] = pd.Categorical(weekday_seasonality["Order Weekday"], categories=weekday_order, ordered=True)
weekday_seasonality = weekday_seasonality.sort_values("Order Weekday")

year_month_sales = df.groupby(["Order Year", "Order Month Name"], as_index=False)["Sales"].sum()
year_month_sales["Order Month Name"] = pd.Categorical(year_month_sales["Order Month Name"], categories=month_order, ordered=True)
year_month_pivot = year_month_sales.pivot(index="Order Year", columns="Order Month Name", values="Sales").reindex(columns=month_order)

weak_months = monthly_trend.nsmallest(5, "monthly_sales")[["YearMonth", "monthly_sales", "monthly_profit"]]
weak_profit_months = monthly_trend.nsmallest(5, "monthly_profit")[["YearMonth", "monthly_sales", "monthly_profit"]]

fig, axes = plt.subplots(1, 3, figsize=(22, 5))
sns.barplot(data=monthly_seasonality, x="Order Month Name", y="total_sales", ax=axes[0], palette="viridis")
axes[0].set_title("Total Sales By Month Name")
axes[0].tick_params(axis="x", rotation=45)

sns.barplot(data=weekday_seasonality, x="Order Weekday", y="total_sales", ax=axes[1], palette="magma")
axes[1].set_title("Total Sales By Weekday")
axes[1].tick_params(axis="x", rotation=45)

sns.heatmap(year_month_pivot, cmap="YlGnBu", ax=axes[2])
axes[2].set_title("Monthly Sales Heatmap By Year")

plt.tight_layout()
plt.show()

display(monthly_seasonality)
display(weekday_seasonality)
display(weak_months)
display(weak_profit_months)

C:\Users\ahmad\AppData\Local\Temp\ipykernel_45160\2344540534.py:28: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=monthly_seasonality, x="Order Month Name", y="total_sales", ax=axes[0], palette="viridis")


C:\Users\ahmad\AppData\Local\Temp\ipykernel_45160\2344540534.py:32: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=weekday_seasonality, x="Order Weekday", y="total_sales", ax=axes[1], palette="magma")


C:\Users\ahmad\AppData\Local\Temp\ipykernel_45160\2344540534.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Order Month Name,average_sales,total_sales,total_profit
4,January,249.146550,94924.8356,9134.4461
3,February,199.170838,59751.2514,10294.6107
7,March,294.548116,205005.4888,28594.6872
0,April,206.230731,137762.1286,11587.4363
8,May,210.923553,155028.8117,22411.3078
6,June,212.996763,152718.6793,21285.7954
5,July,207.377601,147238.0970,13832.6648
1,August,225.274877,159044.0630,21776.9384
11,September,222.451154,307649.9457,36857.4753
10,October,244.594609,200322.9847,31784.0413


,Order Weekday,average_sales,total_sales,total_profit
1,Monday,229.255914,428937.8157,51511.1146
5,Tuesday,259.683094,287209.5022,35570.3288
6,Wednesday,237.350337,88056.9752,14703.8707
4,Thursday,220.072302,321965.7785,39683.7512
0,Friday,235.254268,427692.2593,46520.8242
2,Saturday,216.304944,357984.6830,41277.6202
3,Sunday,225.353127,385353.8464,57129.5120


,YearMonth,monthly_sales,monthly_profit
1,2014-02,4519.8920,862.3084
13,2015-02,11951.4110,2813.8508
0,2014-01,14236.8950,2450.1907
12,2015-01,18174.0756,-3281.0070
24,2016-01,18542.4910,2824.8233


,YearMonth,monthly_sales,monthly_profit
12,2015-01,18174.0756,-3281.0070
6,2014-07,33946.3930,-841.4826
2,2014-03,55691.0090,498.7299
1,2014-02,4519.8920,862.3084
39,2017-04,36521.5361,933.2900


## Bivariate / Multivariate AnalysisThis section compares category and sub-category performance using more than revenue alone. Looking at sales, profit, quantity, and average discount together helps separate headline winners from margin-risk product lines.

In [9]:
category_summary = df.groupby("Category", as_index=False).agg(
    total_sales=("Sales", "sum"),
    total_profit=("Profit", "sum"),
    total_quantity=("Quantity", "sum"),
    average_discount=("Discount", "mean"),
)
category_summary["profit_margin_pct"] = np.where(category_summary["total_sales"] != 0, (category_summary["total_profit"] / category_summary["total_sales"]) * 100, np.nan)
category_summary = category_summary.sort_values("total_sales", ascending=False)

subcategory_summary = df.groupby(["Category", "Sub-Category"], as_index=False).agg(
    total_sales=("Sales", "sum"),
    total_profit=("Profit", "sum"),
    total_quantity=("Quantity", "sum"),
    average_discount=("Discount", "mean"),
)
subcategory_summary["profit_margin_pct"] = np.where(subcategory_summary["total_sales"] != 0, (subcategory_summary["total_profit"] / subcategory_summary["total_sales"]) * 100, np.nan)
top_subcategories = subcategory_summary.sort_values("total_sales", ascending=False).head(10)
lowest_profit_subcategories = subcategory_summary.sort_values("total_profit").head(10)

fig, axes = plt.subplots(1, 3, figsize=(22, 5))
sns.barplot(data=category_summary, x="Category", y="total_sales", ax=axes[0], palette="Set2")
axes[0].set_title("Category Sales")

sns.barplot(data=category_summary, x="Category", y="total_profit", ax=axes[1], palette="Set1")
axes[1].set_title("Category Profit")

sns.barplot(data=top_subcategories, x="Sub-Category", y="total_sales", hue="Category", ax=axes[2])
axes[2].set_title("Top 10 Sub-Categories By Sales")
axes[2].tick_params(axis="x", rotation=60)

plt.tight_layout()
plt.show()

display(category_summary)
display(top_subcategories)
display(lowest_profit_subcategories)

C:\Users\ahmad\AppData\Local\Temp\ipykernel_45160\3275072184.py:21: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=category_summary, x="Category", y="total_sales", ax=axes[0], palette="Set2")


C:\Users\ahmad\AppData\Local\Temp\ipykernel_45160\3275072184.py:24: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=category_summary, x="Category", y="total_profit", ax=axes[1], palette="Set1")


C:\Users\ahmad\AppData\Local\Temp\ipykernel_45160\3275072184.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Category,total_sales,total_profit,total_quantity,average_discount,profit_margin_pct
2,Technology,836154.0330,145454.9481,6939,0.132323,17.395712
0,Furniture,741999.7953,18451.2728,8028,0.173923,2.486695
1,Office Supplies,719047.0320,122490.8008,22906,0.157285,17.035158


,Category,Sub-Category,total_sales,total_profit,total_quantity,average_discount,profit_margin_pct
16,Technology,Phones,330007.0540,44515.7306,3289,0.154556,13.489327
1,Furniture,Chairs,328449.1030,26590.1663,2356,0.170178,8.095673
11,Office Supplies,Storage,223843.6080,21278.8264,3158,0.074704,9.506113
3,Furniture,Tables,206965.5320,-17725.4811,1241,0.261285,-8.564460
6,Office Supplies,Binders,203412.7330,30221.7633,5974,0.372292,14.857361
15,Technology,Machines,189238.6310,3384.7569,440,0.306087,1.788618
13,Technology,Accessories,167380.3180,41936.6357,2976,0.078452,25.054700
14,Technology,Copiers,149528.0300,55617.8249,234,0.161765,37.195585
0,Furniture,Bookcases,114879.9963,-3472.5560,868,0.211140,-3.022768
4,Office Supplies,Appliances,107532.1610,18138.0054,1729,0.166524,16.867517


,Category,Sub-Category,total_sales,total_profit,total_quantity,average_discount,profit_margin_pct
3,Furniture,Tables,206965.5320,-17725.4811,1241,0.261285,-8.564460
0,Furniture,Bookcases,114879.9963,-3472.5560,868,0.211140,-3.022768
12,Office Supplies,Supplies,46673.5380,-1189.0995,647,0.076842,-2.547695
8,Office Supplies,Fasteners,3024.2800,949.5182,914,0.082028,31.396504
15,Technology,Machines,189238.6310,3384.7569,440,0.306087,1.788618
9,Office Supplies,Labels,12486.3120,5546.2540,1400,0.068681,44.418672
5,Office Supplies,Art,27118.7920,6527.7870,3000,0.074874,24.071083
7,Office Supplies,Envelopes,16476.4020,6964.1767,906,0.080315,42.267582
2,Furniture,Furnishings,91705.1640,13059.1436,3563,0.138349,14.240358
4,Office Supplies,Appliances,107532.1610,18138.0054,1729,0.166524,16.867517


## Feature-Specific InsightsRevenue performance usually reflects several interacting forces. This block looks at region, segment, discount, shipping delay, and line-item economics so we can identify which dimensions are associated with stronger revenue or weaker profitability.

In [10]:
region_summary = df.groupby("Region", as_index=False).agg(
    total_sales=("Sales", "sum"),
    total_profit=("Profit", "sum"),
    order_lines=("Order ID", "size"),
).sort_values("total_sales", ascending=False)

segment_summary = df.groupby("Segment", as_index=False).agg(
    total_sales=("Sales", "sum"),
    total_profit=("Profit", "sum"),
    average_discount=("Discount", "mean"),
).sort_values("total_sales", ascending=False)

discount_summary = df.groupby("Discount Bucket", dropna=False, as_index=False).agg(
    total_sales=("Sales", "sum"),
    total_profit=("Profit", "sum"),
    avg_profit_margin=("Profit Margin Pct", "mean"),
    row_count=("Order ID", "size"),
).sort_values("total_sales", ascending=False)

driver_corr = df[["Sales", "Profit", "Quantity", "Discount", "Ship Delay Days", "Profit Margin Pct"]].corr(numeric_only=True)
sample_df = df.sample(min(len(df), 5000), random_state=42).copy()

fig, axes = plt.subplots(1, 3, figsize=(22, 5))
sns.barplot(data=region_summary, x="Region", y="total_sales", ax=axes[0], palette="rocket")
axes[0].set_title("Regional Revenue Contribution")

sns.barplot(data=discount_summary, x="Discount Bucket", y="total_profit", ax=axes[1], palette="coolwarm")
axes[1].set_title("Profit By Discount Bucket")
axes[1].tick_params(axis="x", rotation=30)

sns.scatterplot(data=sample_df, x="Discount", y="Profit", hue="Category", alpha=0.35, ax=axes[2])
axes[2].set_title("Discount vs Profit")

plt.tight_layout()
plt.show()

display(region_summary)
display(segment_summary)
display(discount_summary)
display(driver_corr)

C:\Users\ahmad\AppData\Local\Temp\ipykernel_45160\3911144439.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  discount_summary = df.groupby("Discount Bucket", dropna=False, as_index=False).agg(
C:\Users\ahmad\AppData\Local\Temp\ipykernel_45160\3911144439.py:24: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=region_summary, x="Region", y="total_sales", ax=axes[0], palette="rocket")


C:\Users\ahmad\AppData\Local\Temp\ipykernel_45160\3911144439.py:27: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=discount_summary, x="Discount Bucket", y="total_profit", ax=axes[1], palette="coolwarm")


C:\Users\ahmad\AppData\Local\Temp\ipykernel_45160\3911144439.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Region,total_sales,total_profit,order_lines
3,West,725457.8245,108418.4489,3203
1,East,678781.2400,91522.7800,2848
0,Central,501239.8908,39706.3625,2323
2,South,391721.9050,46749.4303,1620


,Segment,total_sales,total_profit,average_discount
0,Consumer,1.161401e+06,134119.2092,0.158141
1,Corporate,7.061464e+05,91979.1340,0.158228
2,Home Office,4.296531e+05,60298.6785,0.147128


,Discount Bucket,total_sales,total_profit,avg_profit_margin,row_count
0,0%,1.087908e+06,320987.6032,34.016048,4798
2,10-20%,7.921529e+05,91756.2975,17.483863,3709
3,20-40%,2.341379e+05,-35817.4655,-16.685544,460
4,40%+,1.286323e+05,-99558.5905,-108.900322,933
1,0-10%,5.436935e+04,9029.1770,15.579196,94


,Sales,Profit,Quantity,Discount,Ship Delay Days,Profit Margin Pct
Sales,1.000000,0.479064,0.200795,-0.028190,-0.007354,0.003444
Profit,0.479064,1.000000,0.066253,-0.219487,-0.004649,0.223732
Quantity,0.200795,0.066253,1.000000,0.008623,0.018298,-0.005280
Discount,-0.028190,-0.219487,0.008623,1.000000,0.000408,-0.864452
Ship Delay Days,-0.007354,-0.004649,0.018298,0.000408,1.000000,-0.011815
Profit Margin Pct,0.003444,0.223732,-0.005280,-0.864452,-0.011815,1.000000


## Statistical ChecksA simple trend check quantifies whether sales drift upward or downward through time after we smooth short-term noise with the rolling-average analysis.

In [11]:
from scipy import stats
trend_check_df = (
    daily_trend[['Order Date', 'daily_sales']]
    .dropna()
    .sort_values('Order Date')
    .reset_index(drop=True)
)
trend_check_df['day_index'] = np.arange(len(trend_check_df))
pearson_r, pearson_p = stats.pearsonr(trend_check_df['day_index'], trend_check_df['daily_sales'])
monthly_sales = df.groupby('Order Month Name', as_index=False)['Sales'].sum()
monthly_cv = monthly_sales['Sales'].std() / monthly_sales['Sales'].mean()
print(f'Pearson r (time vs daily sales): r={pearson_r:.4f}, p={pearson_p:.2e}')
print(f'Monthly sales coefficient of variation: {monthly_cv:.3f}')

Pearson r (time vs daily sales): r=0.1722, p=1.08e-09
Monthly sales coefficient of variation: 0.480


## Visual AnalysisMoving averages smooth short-term volatility and make it easier to judge whether the business is improving, flattening, or weakening. This block compares recent daily sales with short and long moving averages and highlights the weakest monthly periods observed in the dataset.

In [12]:
recent_window = daily_trend.dropna(subset=["sales_ma_30", "sales_ma_90"]).copy()
latest_signal = recent_window.iloc[-1].to_dict() if not recent_window.empty else {}
trend_signal = "insufficient_history"
if latest_signal:
    if latest_signal["sales_ma_30"] > latest_signal["sales_ma_90"]:
        trend_signal = "short_term_above_long_term"
    elif latest_signal["sales_ma_30"] < latest_signal["sales_ma_90"]:
        trend_signal = "short_term_below_long_term"
    else:
        trend_signal = "short_term_equal_long_term"

below_ma_periods = daily_trend[daily_trend["daily_sales"] < daily_trend["sales_ma_30"]].copy()
weak_period_summary = pd.Series({
    "trend_signal": trend_signal,
    "days_below_30d_ma": int(len(below_ma_periods)),
    "lowest_month_sales": float(monthly_trend["monthly_sales"].min()),
    "lowest_month_profit": float(monthly_trend["monthly_profit"].min()),
    "lowest_month_id": monthly_trend.loc[monthly_trend["monthly_sales"].idxmin(), "YearMonth"],
    "lowest_profit_month_id": monthly_trend.loc[monthly_trend["monthly_profit"].idxmin(), "YearMonth"],
}).to_frame(name="value")

display(weak_period_summary)

,value
trend_signal,short_term_below_long_term
days_below_30d_ma,796
lowest_month_sales,4519.892
lowest_month_profit,-3281.007
lowest_month_id,2014-02
lowest_profit_month_id,2015-01


## Key FindingsThe next cell turns the computed tables into concise business-facing takeaways. The notebook does not hardcode findings in advance; it derives them from the actual results produced above.

In [13]:
top_category_sales = category_summary.iloc[0]
top_category_profit = category_summary.sort_values("total_profit", ascending=False).iloc[0]
weakest_sales_month = monthly_trend.loc[monthly_trend["monthly_sales"].idxmin()]
weakest_profit_month = monthly_trend.loc[monthly_trend["monthly_profit"].idxmin()]
top_region = region_summary.iloc[0]
worst_discount_bucket = discount_summary.sort_values("total_profit").iloc[0]
top_subcategory = top_subcategories.iloc[0]

insights = [
    f"Top revenue category: {top_category_sales['Category']} with sales of {top_category_sales['total_sales']:.2f}.",
    f"Top profit category: {top_category_profit['Category']} with profit of {top_category_profit['total_profit']:.2f}.",
    f"Top revenue sub-category: {top_subcategory['Sub-Category']} with sales of {top_subcategory['total_sales']:.2f}.",
    f"Weakest month by sales: {weakest_sales_month['YearMonth']} with sales of {weakest_sales_month['monthly_sales']:.2f}.",
    f"Weakest month by profit: {weakest_profit_month['YearMonth']} with profit of {weakest_profit_month['monthly_profit']:.2f}.",
    f"Highest revenue region: {top_region['Region']} with sales of {top_region['total_sales']:.2f}.",
    f"Lowest-profit discount bucket: {worst_discount_bucket['Discount Bucket']} with total profit of {worst_discount_bucket['total_profit']:.2f}.",
    f"Current moving-average signal: {trend_signal.replace('_', ' ')}.",
]

for item in insights:
    print(f"- {item}")

- Top revenue category: Technology with sales of 836154.03.
- Top profit category: Technology with profit of 145454.95.
- Top revenue sub-category: Phones with sales of 330007.05.
- Weakest month by sales: 2014-02 with sales of 4519.89.
- Weakest month by profit: 2015-01 with profit of -3281.01.
- Highest revenue region: West with sales of 725457.82.
- Lowest-profit discount bucket: 40%+ with total profit of -99558.59.
- Current moving-average signal: short term below long term.


## Limitations
- This is a descriptive analysis of one sample retail dataset, not a causal experiment.
- Revenue and profit patterns can differ sharply, so sales growth should not be treated as automatically healthy.
- The dataset reflects one specific business context and time span; patterns may not transfer directly to other retailers.
- Some operational drivers such as marketing spend, stockouts, and competitor activity are not present in the workbook.

## Recommendations / Next Steps
- Investigate why the weakest profit months underperform even when sales are not the lowest.
- Review discount-heavy segments and sub-categories for margin leakage before expanding promotions.
- Use high-performing periods and categories as inputs for forecasting, assortment planning, and campaign timing.
- Extend this notebook with customer cohort analysis or region-level seasonality comparisons if deeper planning is needed.

## Common Mistakes
- Ranking product lines by revenue alone and ignoring profit or discount exposure.
- Treating monthly peaks as sustainable growth without checking moving averages.
- Calling a weak month seasonal after seeing only a single low observation.
- Ignoring the possibility that aggressive discounts raise sales while damaging profitability.

## Mini Challenge
1. Repeat the monthly trend analysis at the region level and identify whether weak periods are concentrated in one geography.
2. Build a category-by-quarter view and compare revenue winners with profit winners.
3. Test whether higher-discount buckets consistently deliver weaker average profit margins across all categories.

## Final Summary / Key Takeaways
This notebook builds a complete retail sales trend analysis workflow from transactional data to business insights. It covers long-run sales direction, seasonality, category and sub-category performance, revenue and margin drivers, moving averages, and weak-period detection. The main discipline is simple: validate the data first, compare revenue against profit second, and only then turn patterns into decisions.